# 02 — Column Cleaning

**Purpose:** For each source table: drop constant/redundant columns, apply minimal normalization (units, filters), and save one clean CSV per table.  
**Input:** `data/raw/*.csv` (9 files)  
**Output:** `data/processed/cleaned/` — one `*_clean.csv` per source table  

**Rule:** No feature engineering here. Only structural cleaning (dropping, renaming, unit fixes).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

RAW_DIR   = Path('../data/raw')
CLEAN_DIR = Path('../data/processed/cleaned')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Raw Tables

In [ ]:
FILE_MAP = {
    'world_imports'     : '01_comtrade_world_imports_cleaned.csv',
    'algeria_production': '02_algeria_resources_fao_cleaned.csv',
    'algeria_inputs'    : '03_algeria_resources_fao_inputs_cleaned.csv',
    'algeria_land'      : '04_algeria_resources_fao_land_cleaned.csv',
    'algeria_prices'    : '05_algeria_resources_fao_prices_cleaned.csv',
    'algeria_worldbank' : '06_algeria_resources_worldbank_cleaned.csv',
    'unit_values'       : '09_comtrade_unit_values_cleaned.csv',
    'country_metadata'  : '11_country_metadata_cleaned.csv',
    'algeria_exports'   : '12_algeria_exports_comtrade_cleaned.csv',
}

dfs = {}
for name, fname in FILE_MAP.items():
    dfs[name] = pd.read_csv(RAW_DIR / fname, low_memory=False)
    print(f"✅  {name:<22} {len(dfs[name]):>8,} rows  {len(dfs[name].columns):>3} cols")

## 2. world_imports

**Source:** UN Comtrade — global import flows  
**What to drop:** Administrative constants, redundant codes, less-granular HS4 code

In [ ]:
df = dfs['world_imports'].copy()

# Verify quantity_alt vs net_weight_kg before deciding which to keep
n_differ = (df['quantity_alt'] != df['net_weight_kg']).sum()
print(f"quantity_alt ≠ net_weight_kg in {n_differ:,} rows  "
      f"({n_differ/len(df)*100:.1f}%) — difference explained by energy rows (kWh)")
print(f"quantity_unit distribution:\n{df['quantity_unit'].value_counts()}")

# Drop constants and redundant columns
# flow_code/flow_desc/classification/customs_proc_code → always same value
# importer_code/exporter_code → have ISO3 names already
# hs_code_4digit → less granular than hs_code_6digit
drop_cols = [
    'flow_code', 'flow_desc', 'classification', 'customs_proc_code',
    'importer_code', 'exporter_code',
    'hs_code_4digit',
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

dfs['world_imports'] = df
print(f"\nworld_imports cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Remaining columns: {df.columns.tolist()}")

## 3. algeria_production

**Source:** FAOSTAT — Algeria production, yield, area harvested per crop  
**What to drop:** Area codes (Algeria only), item codes redundant with name, Year Code = Year

In [ ]:
df = dfs['algeria_production'].copy()

# Year Code is always equal to Year — confirmed constant duplicate
if 'Year Code' in df.columns and 'Year' in df.columns:
    assert (df['Year Code'] == df['Year']).all(), "Year Code ≠ Year — investigate!"

drop_cols = [
    'Area Code', 'Area Code (M49)', 'Area',   # Algeria only → constant
    'Item Code', 'Item Code (CPC)',            # redundant with Item name
    'Element Code',                            # redundant with Element name
    'Year Code',                               # identical to Year
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

# Keep only elements needed for features
keep_elements = ['Production', 'Area harvested', 'Yield', 'Yield/Carcass Weight']
df = df[df['Element'].isin(keep_elements)]

dfs['algeria_production'] = df
print(f"algeria_production cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Elements kept: {df['Element'].unique()}")

## 4. algeria_inputs

**Source:** FAOSTAT — Algeria fertilizer use and imports  
**What to drop:** Area codes (Algeria only), element/item codes

In [ ]:
df = dfs['algeria_inputs'].copy()

drop_cols = [
    'Area Code', 'Area Code (M49)', 'Area',
    'Item Code', 'Element Code', 'Year Code',
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

# Keep only elements needed: total fertilizer use and import quantity
keep_elements = ['Agricultural Use', 'Import quantity']
df = df[df['Element'].isin(keep_elements)]

dfs['algeria_inputs'] = df
print(f"algeria_inputs cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")

## 5. algeria_land

**Source:** FAOSTAT — Algeria agricultural land statistics

In [ ]:
df = dfs['algeria_land'].copy()

drop_cols = [
    'Area Code', 'Area Code (M49)', 'Area',
    'Item Code', 'Element Code', 'Year Code',
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

keep_elements = [
    'Area',
    'Share in Agricultural land',
    'Value of agricultural production (Int. $) per Area',
]
df = df[df['Element'].isin(keep_elements)]

dfs['algeria_land'] = df
print(f"algeria_land cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")

## 6. algeria_prices

**Source:** FAOSTAT — Algeria producer prices  
**What to drop:** SLC prices (local currency, redundant with USD), monthly rows (keep annual only)

In [ ]:
df = dfs['algeria_prices'].copy()

# Keep annual values only (monthly breakdown not needed)
if 'Months' in df.columns:
    df = df[df['Months'] == 'Annual value']

# Drop SLC (local currency) — redundant with USD version
df = df[df['Element'] != 'Producer Price (SLC/tonne)']

drop_cols = [
    'Area Code', 'Area Code (M49)', 'Area',
    'Item Code', 'Item Code (CPC)',
    'Element Code', 'Year Code',
    'Months Code', 'Months',
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

dfs['algeria_prices'] = df
print(f"algeria_prices cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Elements: {df['Element'].unique()}")

## 7. algeria_worldbank

**Source:** World Bank — Algeria macroeconomic indicators  
**What to drop:** Country code/name (Algeria only), indicator code (have label), decimal

In [ ]:
df = dfs['algeria_worldbank'].copy()

drop_cols = [
    'country_code', 'country_name',  # Algeria only — constant
    'wb_indicator', 'indicator_name', # redundant with indicator_label
    'decimal',                        # formatting metadata
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

dfs['algeria_worldbank'] = df
print(f"algeria_worldbank cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Columns: {df.columns.tolist()}")

## 8. unit_values

**Source:** UN Comtrade — world-average unit values (USD/kg) per product per year  
**Note:** Contains both `exporter_name = 'World'` (average) and bilateral rows. We keep both for reference — the integration step will extract only World rows.

In [ ]:
df = dfs['unit_values'].copy()

world_rows     = (df['exporter_name'] == 'World').sum()
bilateral_rows = (df['exporter_name'] != 'World').sum()
print(f"World avg rows : {world_rows:,}")
print(f"Bilateral rows : {bilateral_rows:,}")
print(f"Columns        : {df.columns.tolist()}")

# No columns to drop — table is already compact
dfs['unit_values'] = df
print(f"\nunit_values: no changes needed")

## 9. country_metadata

**Source:** World Bank — importer country profiles  
**What to drop:** `country_code` (redundant with iso3), `wb_indicator` (redundant with label), `year` (single snapshot)

In [ ]:
df = dfs['country_metadata'].copy()

# Fix Namibia: ISO code 'NA' reads as NaN in pandas
df.loc[df['country_name'] == 'Namibia', 'country_code'] = 'NA'

drop_cols = [
    'country_code',   # redundant with country_iso3
    'wb_indicator',   # redundant with indicator_label
    'year',           # single cross-sectional snapshot
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

dfs['country_metadata'] = df
print(f"country_metadata cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Columns: {df.columns.tolist()}")
print(f"Indicators available: {df['indicator_label'].unique()}")

## 10. algeria_exports

**Source:** UN Comtrade — Algeria's actual export flows  
**What to drop:** Reporter columns (always Algeria), HS4 code, flow/classification constants  
**Unit fix:** Normalize tonnes → kg so unit_value calculations are consistent

In [ ]:
df = dfs['algeria_exports'].copy()

# Keep exports only (drop re-imports if present)
if 'flow_desc' in df.columns:
    df = df[df['flow_desc'] == 'Exports']

# Normalize tonnes → kg for consistency with world_imports
if 'quantity_unit' in df.columns and 'net_weight_kg' in df.columns:
    tonne_mask = df['quantity_unit'] == 'tonnes'
    df.loc[tonne_mask, 'net_weight_kg'] = df.loc[tonne_mask, 'net_weight_kg'] * 1000
    if 'quantity_alt' in df.columns:
        df.loc[tonne_mask, 'quantity_alt'] = df.loc[tonne_mask, 'quantity_alt'] * 1000
    df.loc[tonne_mask, 'quantity_unit'] = 'kg'
    print(f"Converted {tonne_mask.sum():,} rows from tonnes to kg")

drop_cols = [
    'reporter_code', 'reporter_name', 'reporter_iso3',  # always Algeria
    'hs_code_4digit', 'hs4_oec_id',                     # less granular
    'flow_code', 'flow_desc', 'classification',          # constants
    'period',                                            # duplicate of year
]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

dfs['algeria_exports'] = df
print(f"algeria_exports cleaned: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Columns: {df.columns.tolist()}")

## 11. Save Cleaned Tables

In [ ]:
for name, df in dfs.items():
    out_path = CLEAN_DIR / f'{name}_clean.csv'
    df.to_csv(out_path, index=False)
    print(f"✅  {name:<22} saved → {df.shape[0]:,} rows × {df.shape[1]} cols")

print(f"\nAll cleaned tables saved to: {CLEAN_DIR.resolve()}")

---
**Next:** `03_integration_and_features.ipynb` — merge tables, engineer features, encode categoricals.